# KaroSpace Feature Distribution Calculations

This notebook recreates the tables shown in:

- `Exploration > Features > Distribution`
- `Statistics > Features > Distribution`

By default this notebook applies `RC` normalization: library-size normalization without `log1p`. Set `statistics_normalization = "LogNormalize"` or `statistics_normalized_layer = "data"` to match those KaroSpace export options.

In [128]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

## Plain variables

In [ ]:
h5ad_path = "./data.h5ad"

assay = "rna"  # "rna" or "protein"
protein_matrix_key = "protein"
protein_feature_table_key = "protein_var"

annotation_col = "leiden_rna"
replicate_col = "sample_id"  # Used only for the pseudobulk Statistics table

rna_genes = ["ABCA1", "ACSL4", "ADAM28", "ADAM9", "ADAMTS1"]
protein_genes = ["CD1c - TRITC"]
genes = protein_genes if assay == "protein" else rna_genes
n_cells_downsample = None  # None = all cells. Use an integer to downsample.
random_seed = 7

# Distribution display values. Defaults match KaroSpace: RC from statistics_counts_layer.
statistics_counts_layer = None  # None = adata.X. Use "counts" to use adata.layers["counts"].
statistics_normalization = "RC"  # "RC" or "LogNormalize"
statistics_scale_factor = 10000.0  # Used by RC and LogNormalize.
statistics_normalized_layer = None  # Example: "data". If set, use this layer directly.
statistics_min_cell_counts = 10  # Same as --statistics-min-cell-counts. 0 disables the filter.
statistics_min_feature_counts = 0  # Same as --statistics-min-feature-counts. Applied to selected features here.
pseudobulk_min_cells_per_pseudobulk = 20  # Same as --pseudobulk-min-cells-per-pseudobulk.


## Load data and select cells/features

In [130]:
adata = ad.read_h5ad(h5ad_path)

if assay == "rna":
    assay_matrix = adata.X
    assay_source = "adata.X"
    assay_layers = adata.layers
    feature_names = [str(x) for x in adata.var_names]
elif assay == "protein":
    assay_matrix = adata.obsm[protein_matrix_key]
    assay_source = f"adata.obsm[{protein_matrix_key!r}]"
    assay_layers = {protein_matrix_key: assay_matrix}
    if f"{protein_matrix_key}_arcsinh" in adata.obsm:
        assay_layers[f"{protein_matrix_key}_arcsinh"] = adata.obsm[f"{protein_matrix_key}_arcsinh"]
    protein_var = adata.uns[protein_feature_table_key]
    feature_names = [str(x) for x in protein_var.iloc[:, 0].to_numpy()]
else:
    raise ValueError('assay must be "rna" or "protein"')

def dense(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)

def matrix_for_layer(layer_name):
    if layer_name is None or layer_name == "X":
        return assay_matrix, assay_source
    if layer_name in assay_layers:
        return assay_layers[layer_name], f"{assay} matrix {layer_name!r}"
    if layer_name in adata.layers:
        return adata.layers[layer_name], f"adata.layers[{layer_name!r}]"
    if layer_name in adata.obsm:
        return adata.obsm[layer_name], f"adata.obsm[{layer_name!r}]"
    return assay_matrix, f"{assay_source} (matrix {layer_name!r} not found)"

def take_columns(matrix, rows, cols):
    return dense(matrix[rows, :][:, cols]).astype(float, copy=False)

def library_normalized_columns(matrix, rows, cols, target_sum):
    values = take_columns(matrix, rows, cols)
    totals = np.asarray(matrix[rows, :].sum(axis=1), dtype=float).ravel()
    scale = np.divide(target_sum, totals, out=np.zeros_like(totals, dtype=float), where=totals > 0)
    return values * scale[:, None]

def distribution_columns(rows, cols):
    if statistics_normalized_layer:
        matrix, source = matrix_for_layer(statistics_normalized_layer)
        return take_columns(matrix, rows, cols), f"{source}, no extra normalization"

    matrix, source = matrix_for_layer(statistics_counts_layer)
    mode = str(statistics_normalization).strip().lower()
    if mode == "rc":
        values = library_normalized_columns(matrix, rows, cols, statistics_scale_factor)
        return values, f"RC {source}, scale_factor={statistics_scale_factor:g}, no log1p"
    if mode in {"lognormalize", "log_normalize", "log-normalize"}:
        values = library_normalized_columns(matrix, rows, cols, statistics_scale_factor)
        return np.log1p(values), f"LogNormalize {source}, target_sum={statistics_scale_factor:g}, log1p"
    raise ValueError('statistics_normalization must be RC or LogNormalize')

if not genes:
    genes = feature_names[:5]
genes = [str(g) for g in genes[:5]]
feature_pos = {name: i for i, name in enumerate(feature_names)}
missing = [g for g in genes if g not in feature_pos]
if missing:
    raise ValueError(f"Features not found in adata.var_names: {missing}")
cols = [feature_pos[g] for g in genes]

valid = adata.obs[annotation_col].notna().to_numpy().copy()
if replicate_col:
    valid &= adata.obs[replicate_col].notna().to_numpy()
cell_idx = np.flatnonzero(valid)
if n_cells_downsample is not None and len(cell_idx) > int(n_cells_downsample):
    rng = np.random.default_rng(random_seed)
    cell_idx = np.sort(rng.choice(cell_idx, size=int(n_cells_downsample), replace=False))

obs = adata.obs.iloc[cell_idx].copy()
labels = obs[annotation_col].astype(str)
if pd.api.types.is_categorical_dtype(obs[annotation_col]):
    categories = [str(c) for c in obs[annotation_col].cat.categories if (labels == str(c)).any()]
else:
    categories = sorted(labels.unique())

exploration_values, exploration_source = distribution_columns(cell_idx, cols)
statistics_values, statistics_source = distribution_columns(cell_idx, cols)

exploration_df = pd.DataFrame(exploration_values, index=obs.index, columns=genes)
statistics_df = pd.DataFrame(statistics_values, index=obs.index, columns=genes)

statistics_count_matrix, statistics_filter_source = matrix_for_layer(statistics_counts_layer)
if int(statistics_min_cell_counts) > 0:
    statistics_cell_totals = np.asarray(statistics_count_matrix[cell_idx, :].sum(axis=1), dtype=float).ravel()
    statistics_cell_mask = np.isfinite(statistics_cell_totals) & (statistics_cell_totals >= int(statistics_min_cell_counts))
else:
    statistics_cell_mask = np.ones(len(cell_idx), dtype=bool)
statistics_obs = obs.iloc[statistics_cell_mask].copy()
statistics_labels = statistics_obs[annotation_col].astype(str)
statistics_raw_values = pd.DataFrame(
    take_columns(statistics_count_matrix, cell_idx[statistics_cell_mask], cols),
    index=statistics_obs.index,
    columns=genes,
)
if int(statistics_min_feature_counts) > 0:
    statistics_gene_totals = statistics_raw_values.sum(axis=0)
    statistics_genes = [gene for gene in genes if statistics_gene_totals[gene] >= int(statistics_min_feature_counts)]
else:
    statistics_genes = list(genes)
if not statistics_genes:
    raise ValueError("No selected features pass statistics_min_feature_counts")
statistics_df_filtered = statistics_df.iloc[statistics_cell_mask][statistics_genes].copy()
if pd.api.types.is_categorical_dtype(statistics_obs[annotation_col]):
    statistics_categories = [str(c) for c in statistics_obs[annotation_col].cat.categories if (statistics_labels == str(c)).any()]
else:
    statistics_categories = sorted(statistics_labels.unique())

print(f"Cells used: {len(obs):,}")
print(f"Features used: {genes}")
print(f"Exploration source: {exploration_source}")
print(f"Statistics source: {statistics_source}")
print(f"Statistics count-filter source: {statistics_filter_source}")
print(f"Statistics cells after count filter: {len(statistics_obs):,}")
print(f"Statistics features after count filter: {statistics_genes}")
print(f"Categories: {categories}")


Cells used: 46,003
Features used: ['ABCA1', 'ACSL4', 'ADAM28', 'ADAM9', 'ADAMTS1']
Exploration source: RC adata.X, scale_factor=10000, no log1p
Statistics source: RC adata.X, scale_factor=10000, no log1p
Statistics count-filter source: adata.X
Statistics cells after count filter: 44,925
Statistics features after count filter: ['ABCA1', 'ACSL4', 'ADAM28', 'ADAM9', 'ADAMTS1']
Categories: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13']


/var/folders/pb/p96j3wgd6t73npk9p7p6gbjh0000gp/T/ipykernel_50189/510429327.py:76: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(obs[annotation_col]):
/var/folders/pb/p96j3wgd6t73npk9p7p6gbjh0000gp/T/ipykernel_50189/510429327.py:108: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(statistics_obs[annotation_col]):


## Exploration > Features > Distribution

HTML table columns: `Group`, `n`, `Mean`, `Median`, `% Expr`. The default sort is `Mean` descending.

In [123]:
def exploration_distribution_table(gene):
    rows = []
    for category in categories:
        values = exploration_df.loc[labels == category, gene].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        rows.append({
            "Group": category,
            "n": int(len(values)),
            "Mean": values.mean() if len(values) else np.nan,
            "Median": np.median(values) if len(values) else np.nan,
            "% Expr": 100.0 * np.mean(values > 0) if len(values) else np.nan,
        })
    return pd.DataFrame(rows).sort_values("Mean", ascending=False, na_position="last").reset_index(drop=True)

exploration_tables = {gene: exploration_distribution_table(gene) for gene in genes}
feature_to_show = genes[0]
print(f"Exploration table for {feature_to_show}")
display(exploration_tables[feature_to_show])


Exploration table for ABCA1


,Group,n,Mean,Median,% Expr
0,8,3256,17.960060,0.000000,39.772727
1,11,18,15.873016,0.000000,5.555556
2,2,1821,15.675140,0.000000,24.766612
3,0,4444,11.737084,7.446021,58.393339
4,12,820,5.935301,0.000000,0.609756
5,1,6233,5.779881,0.000000,11.519333
6,10,2136,4.931661,0.000000,4.166667
7,6,1423,4.611125,0.000000,3.654252
8,5,589,3.210617,0.000000,0.848896
9,7,6773,2.750250,0.000000,6.703086


## Statistics > Features > Distribution: Wilcoxon method

HTML table columns: `Category`, `Mean`, `Cells`. Background mean is displayed above the table.

In [124]:
def statistics_wilcoxon_table(gene):
    rows = []
    for category in statistics_categories:
        values = statistics_df_filtered.loc[statistics_labels == category, gene].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        rows.append({
            "Category": category,
            "Mean": values.mean() if len(values) else np.nan,
            "Cells": int(len(values)),
        })
    table = pd.DataFrame(rows).sort_values("Mean", ascending=False, na_position="last").reset_index(drop=True)
    return table, statistics_df_filtered[gene].mean()

statistics_wilcoxon_tables = {}
statistics_wilcoxon_background = {}
for gene in statistics_genes:
    table, background = statistics_wilcoxon_table(gene)
    statistics_wilcoxon_tables[gene] = table
    statistics_wilcoxon_background[gene] = background

feature_to_show = feature_to_show if feature_to_show in statistics_genes else statistics_genes[0]
print(f"Statistics/Wilcoxon table for {feature_to_show}. Background mean: {statistics_wilcoxon_background[feature_to_show]:.4f}")
display(statistics_wilcoxon_tables[feature_to_show])


Statistics/Wilcoxon table for ABCA1. Background mean: 5.5884


,Category,Mean,Cells
0,8,17.965578,3255
1,11,16.806723,17
2,2,15.761695,1811
3,0,11.737084,4444
4,1,5.845530,6163
5,10,4.971226,2119
6,6,4.690230,1399
7,5,3.622708,522
8,12,3.276952,468
9,7,2.750250,6773


## Statistics > Features > Distribution: Pseudobulk method

This recreates the displayed category mean table for the Pseudobulk method. DESeq2 itself uses raw-count pseudobulks for marker testing; this table uses display-scale means. The filter order is: remove low-count cells with `statistics_min_cell_counts`, remove low-count selected features with `statistics_min_feature_counts`, remove replicate-category pseudobulk samples with fewer than `pseudobulk_min_cells_per_pseudobulk` cells, then calculate the displayed means.

In [125]:
pseudobulk_obs = statistics_obs.copy()
pseudobulk_values = statistics_df_filtered.copy()
pseudobulk_genes = list(statistics_genes)

def statistics_pseudobulk_table(gene):
    tmp = pseudobulk_obs[[replicate_col, annotation_col]].copy()
    tmp["value"] = pseudobulk_values[gene].to_numpy(dtype=float)
    tmp = tmp[np.isfinite(tmp["value"])]

    per_sample = (
        tmp.groupby([replicate_col, annotation_col], observed=True)["value"]
        .agg(total="sum", cells="size")
        .reset_index()
    )
    per_sample = per_sample[per_sample["cells"] >= int(pseudobulk_min_cells_per_pseudobulk)]
    per_sample["sample_mean"] = per_sample["total"] / per_sample["cells"]

    rows = []
    for category in categories:
        sub = per_sample[per_sample[annotation_col].astype(str) == category]
        rows.append({
            "Category": category,
            "Mean": sub["sample_mean"].mean() if len(sub) else np.nan,
            "Cells": int(sub["cells"].sum()) if len(sub) else 0,
        })
    by_replicate = per_sample.groupby(replicate_col, observed=True).agg(total=("total", "sum"), cells=("cells", "sum"))
    background = (by_replicate["total"] / by_replicate["cells"]).mean()
    table = pd.DataFrame(rows).sort_values("Mean", ascending=False, na_position="last").reset_index(drop=True)
    return table, background

feature_to_show_pseudobulk = feature_to_show if feature_to_show in pseudobulk_genes else pseudobulk_genes[0]

statistics_pseudobulk_tables = {}
statistics_pseudobulk_background = {}
for gene in pseudobulk_genes:
    table, background = statistics_pseudobulk_table(gene)
    statistics_pseudobulk_tables[gene] = table
    statistics_pseudobulk_background[gene] = background

print(f"Statistics/Pseudobulk table for {feature_to_show_pseudobulk}. Background mean: {statistics_pseudobulk_background[feature_to_show_pseudobulk]:.4f}")
display(statistics_pseudobulk_tables[feature_to_show_pseudobulk])


Statistics/Pseudobulk table for ABCA1. Background mean: 7.5573


,Category,Mean,Cells
0,8,18.001560,3245
1,2,15.482731,1811
2,0,11.743281,4438
3,10,7.141734,2119
4,1,4.912463,6163
5,7,3.718482,6758
6,3,3.689557,1281
7,6,3.511480,1381
8,4,2.373449,12431
9,5,2.211929,522


## All five features

In [126]:
exploration_all = pd.concat(exploration_tables, names=["Feature", "row"]).reset_index(level="Feature").reset_index(drop=True)
wilcoxon_all = pd.concat(statistics_wilcoxon_tables, names=["Feature", "row"]).reset_index(level="Feature").reset_index(drop=True)
wilcoxon_all["Background mean"] = wilcoxon_all["Feature"].map(statistics_wilcoxon_background)
pseudobulk_all = pd.concat(statistics_pseudobulk_tables, names=["Feature", "row"]).reset_index(level="Feature").reset_index(drop=True)
pseudobulk_all["Background mean"] = pseudobulk_all["Feature"].map(statistics_pseudobulk_background)

print("Exploration > Features > Distribution")
display(exploration_all)
print("Statistics > Features > Distribution: Wilcoxon")
display(wilcoxon_all)
print("Statistics > Features > Distribution: Pseudobulk display means")
display(pseudobulk_all)


Exploration > Features > Distribution


,Feature,Group,n,Mean,Median,% Expr
0,ABCA1,8,3256,17.960060,0.000000,39.772727
1,ABCA1,11,18,15.873016,0.000000,5.555556
2,ABCA1,2,1821,15.675140,0.000000,24.766612
3,ABCA1,0,4444,11.737084,7.446021,58.393339
4,ABCA1,12,820,5.935301,0.000000,0.609756
...,...,...,...,...,...,...
65,ADAMTS1,8,3256,1.438637,0.000000,2.549140
66,ADAMTS1,4,12450,0.680450,0.000000,1.293173
67,ADAMTS1,7,6773,0.513776,0.000000,1.240219
68,ADAMTS1,9,4232,0.126372,0.000000,0.732514


Statistics > Features > Distribution: Wilcoxon


,Feature,Category,Mean,Cells,Background mean
0,ABCA1,8,17.965578,3255,5.588416
1,ABCA1,11,16.806723,17,5.588416
2,ABCA1,2,15.761695,1811,5.588416
3,ABCA1,0,11.737084,4444,5.588416
4,ABCA1,1,5.845530,6163,5.588416
...,...,...,...,...,...
65,ADAMTS1,8,1.439079,3255,3.239539
66,ADAMTS1,4,0.680997,12440,3.239539
67,ADAMTS1,7,0.513776,6773,3.239539
68,ADAMTS1,9,0.126372,4232,3.239539


Statistics > Features > Distribution: Pseudobulk display means


,Feature,Category,Mean,Cells,Background mean
0,ABCA1,8,18.001560,3245,7.557261
1,ABCA1,2,15.482731,1811,7.557261
2,ABCA1,0,11.743281,4438,7.557261
3,ABCA1,10,7.141734,2119,7.557261
4,ABCA1,1,4.912463,6163,7.557261
...,...,...,...,...,...
65,ADAMTS1,8,1.443514,3245,4.948385
66,ADAMTS1,7,0.475646,6758,4.948385
67,ADAMTS1,9,0.126432,4230,4.948385
68,ADAMTS1,11,NaN,0,4.948385
